# Agent Hook Capabilities

**Purpose:** Prove through code that agent hooks work as documented — that they fire, return decisions,
can access conversation via `transcript_path`, and support model selection.

| # | Question | How we prove it |
|---|----------|----------------|
| 3 | Can agent hooks access conversation history? | Agent hook reads `transcript_path`, extracts a secret word planted in the prompt, returns it in `reason` |
| 4 | What model options are available? | Configure `model` field, observe whether the hook fires without error |

Source: `openspec/changes/rodin/proposal.md` — Outstanding research questions 3, 4

## Test design

### Key lesson from v1-v2

Checking for a marker string in the output doesn't prove anything. The hook's denial reason goes
to **Claude** (the agent), not verbatim to the output JSON. Claude paraphrases. The correct signals are:

1. **Tool blocked**: Claude's response says it was blocked/prevented/denied, and the tool output is absent
2. **Subagent spawned**: `modelUsage` shows a secondary model (the hook's subagent)
3. **Tool ran**: The expected tool output appears in the result (hook allowed or didn't fire)

### Hook types under test

| Type | Mechanism | Proof of firing |
|------|-----------|-----------------|
| `"command"` | Shell script, stdin JSON | Script writes side-effect file |
| `"prompt"` | Single LLM call, returns `{ok, reason}` | Tool is blocked, Claude mentions denial |
| `"agent"` | Subagent with Read/Grep/Glob, returns `{ok, reason}` | Secondary model in `modelUsage`, tool blocked |

**Note on inference strength**: A secondary model appearing in `modelUsage` is *positive evidence*
that a hook's LLM call ran. However, its *absence* does not prove non-firing — Test 1 shows prompt
hooks fire and deny with only a single model visible (the hook model matches the session model, so
`modelUsage` merges their token counts into one entry). For agent hooks, we can observe whether the
tool was denied, but cannot definitively determine from `modelUsage` alone whether the hook
infrastructure invoked the hook.

## Setup

In [ ]:
!claude --version

In [ ]:
import json, shutil
from pathlib import Path
from lib import run_claude

PLUGIN_DIR = Path("/tmp/claude/agent-hook-test")

def create_plugin(hook_type: str, prompt: str, *, model: str | None = None, matcher: str = "Bash"):
    """Create a plugin with a single hook on PreToolUse."""
    if PLUGIN_DIR.exists():
        shutil.rmtree(PLUGIN_DIR)

    (PLUGIN_DIR / ".claude-plugin").mkdir(parents=True)
    (PLUGIN_DIR / ".claude-plugin" / "plugin.json").write_text(json.dumps({
        "name": "hook-test",
        "version": "0.1.0",
        "description": f"Tests {hook_type} hook"
    }))

    hook_entry = {"type": hook_type, "prompt": prompt}
    if model:
        hook_entry["model"] = model

    (PLUGIN_DIR / "hooks").mkdir()
    (PLUGIN_DIR / "hooks" / "hooks.json").write_text(json.dumps({
        "hooks": {
            "PreToolUse": [{
                "matcher": matcher,
                "hooks": [hook_entry]
            }]
        }
    }, indent=2))

    return PLUGIN_DIR

def analyze_result(cr, expected_output: str = "hello") -> dict:
    """Analyze a ClaudeResult for hook behavior signals."""
    output = cr.output if isinstance(cr.output, dict) else json.loads(cr.output) if isinstance(cr.output, str) else {}
    result_text = output.get("result", "")
    model_usage = output.get("modelUsage", {})
    num_turns = output.get("num_turns", 0)

    tool_ran = expected_output in result_text.lower()
    denied = any(w in result_text.lower() for w in ["prevented", "denied", "blocked", "hook", "not allowed"])
    models_used = list(model_usage.keys())
    has_secondary_model = len(models_used) > 1

    return {
        "tool_ran": tool_ran,
        "denied": denied,
        "num_turns": num_turns,
        "models_used": models_used,
        "has_secondary_model": has_secondary_model,
        "result_text": result_text,
        "model_usage": model_usage,
    }

print("Setup ready.")

## Test 1: Prompt hook denies a tool call

Baseline test. Prompt hooks are simpler (single LLM call, no subagent). If this denies,
the hook infrastructure works and we can isolate agent hook behavior from there.

In [ ]:
create_plugin("prompt", "This tool call must be denied. Respond with ok=false and reason: policy violation.")

cr = run_claude("Use Bash to run: echo hello", plugin_dir=PLUGIN_DIR)
r = analyze_result(cr)

print("PROMPT HOOK — DENY TEST")
print("="*60)
print(f"  Tool ran:            {r['tool_ran']}")
print(f"  Denial detected:     {r['denied']}")
print(f"  Turns:               {r['num_turns']}")
print(f"  Models:              {r['models_used']}")
print(f"  Result text:         {r['result_text'][:300]}")

if r['denied'] and not r['tool_ran']:
    print("\n>>> PROVED: Prompt hook fired and denied the tool call")
elif r['tool_ran']:
    print("\n>>> DISPROVED: Tool ran — prompt hook did not deny")
else:
    print("\n>>> INCONCLUSIVE: Tool didn't run but denial not detected in response")

## Test 2: Agent hook denies a tool call

Same test as above but with `type: "agent"`. If the tool is denied, the agent hook subagent fired.
Also check `modelUsage` for a secondary model entry — that's the subagent.

In [ ]:
create_plugin("agent", "This tool call must be denied. Respond with ok=false and reason: policy violation.")

cr = run_claude("Use Bash to run: echo hello", plugin_dir=PLUGIN_DIR)
r = analyze_result(cr)

print("AGENT HOOK — DENY TEST")
print("="*60)
print(f"  Tool ran:            {r['tool_ran']}")
print(f"  Denial detected:     {r['denied']}")
print(f"  Turns:               {r['num_turns']}")
print(f"  Models:              {r['models_used']}")
print(f"  Secondary model:     {r['has_secondary_model']}")
print(f"  Result text:         {r['result_text'][:300]}")

# Show model token counts to identify subagent activity
for model, usage in r['model_usage'].items():
    print(f"\n  {model}:")
    print(f"    input={usage.get('inputTokens',0)} output={usage.get('outputTokens',0)} "
          f"cacheRead={usage.get('cacheReadInputTokens',0)}")

if r['denied'] and not r['tool_ran']:
    print("\n>>> PROVED: Agent hook fired and denied the tool call")
elif r['tool_ran'] and r['has_secondary_model']:
    print("\n>>> PARTIAL: Agent hook subagent spawned (secondary model visible) but ALLOWED the tool")
    print("    The subagent did not follow the deny instruction")
elif r['tool_ran']:
    print("\n>>> NOT DENIED: Tool ran — agent hook did not deny the tool call")
    print("    Whether the hook was invoked but ineffective, or never dispatched, is indeterminate")
else:
    print("\n>>> INCONCLUSIVE")

## Test 3: transcript_path is readable (command hook baseline)

Command hooks are proven to work (hook-interactions notebook). Use one to dump `transcript_path`
contents and verify the file exists and contains session data.

In [ ]:
# Use a command hook to dump transcript_path contents
# This proves transcript_path works regardless of agent hook support

TRANSCRIPT_DUMP = Path("/tmp/claude/transcript-dump.json")
TRANSCRIPT_DUMP.unlink(missing_ok=True)

SCRIPT = Path("/tmp/claude/dump-transcript.py")
SCRIPT.write_text('''#!/usr/bin/env python3
import json, sys
from pathlib import Path

event = json.loads(sys.stdin.read())
transcript_path = event.get("transcript_path", "")
dump_path = Path("/tmp/claude/transcript-dump.json")

result = {"transcript_path": transcript_path, "exists": False, "line_count": 0, "sample": ""}

if transcript_path and Path(transcript_path).exists():
    content = Path(transcript_path).read_text()
    lines = [l for l in content.strip().splitlines() if l.strip()]
    result["exists"] = True
    result["line_count"] = len(lines)
    result["sample"] = content[:2000]

dump_path.write_text(json.dumps(result, indent=2))
print(json.dumps({"continue": True}))
''')
SCRIPT.chmod(0o755)

# Create command hook plugin
if PLUGIN_DIR.exists():
    shutil.rmtree(PLUGIN_DIR)
(PLUGIN_DIR / ".claude-plugin").mkdir(parents=True)
(PLUGIN_DIR / ".claude-plugin" / "plugin.json").write_text(json.dumps({
    "name": "transcript-dump", "version": "0.1.0",
    "description": "Dumps transcript_path contents"
}))
(PLUGIN_DIR / "hooks").mkdir()
(PLUGIN_DIR / "hooks" / "hooks.json").write_text(json.dumps({
    "hooks": {"PreToolUse": [{"matcher": "Bash", "hooks": [{
        "type": "command", "command": str(SCRIPT)
    }]}]}
}, indent=2))

cr = run_claude("List the files in the current directory using Bash: ls", plugin_dir=PLUGIN_DIR, permission_mode="delegate")

print(f"Exit code: {cr.returncode}")

if TRANSCRIPT_DUMP.exists():
    dump = json.loads(TRANSCRIPT_DUMP.read_text())
    print(f"transcript_path: {dump['transcript_path']}")
    print(f"File exists: {dump['exists']}")
    print(f"Line count: {dump['line_count']}")
    print(f"\nTranscript sample (first 1000 chars):")
    print(dump['sample'][:1000])
    print(f"\n>>> PROVED: transcript_path points to a real, readable file")
else:
    print("Dump file not created — command hook didn't fire")

## Test 4: Agent hook with model selection

Test `model` field on agent hooks. Run with haiku and sonnet.
Check for secondary model in `modelUsage` and whether the tool was blocked.

In [ ]:
MODELS = ["haiku", "sonnet"]
model_results = {}

for model_name in MODELS:
    create_plugin("agent",
        "This tool call must be denied. Respond with ok=false and reason: policy violation.",
        model=model_name
    )

    cr = run_claude("Use Bash to run: echo hello", plugin_dir=PLUGIN_DIR)
    r = analyze_result(cr)
    model_results[model_name] = r

    print(f"\nModel: {model_name}")
    print(f"  Tool ran:        {r['tool_ran']}")
    print(f"  Denied:          {r['denied']}")
    print(f"  Models used:     {r['models_used']}")
    print(f"  Secondary model: {r['has_secondary_model']}")
    for m, u in r['model_usage'].items():
        print(f"    {m}: in={u.get('inputTokens',0)} out={u.get('outputTokens',0)}")

print("\n" + "="*60)
print("MODEL SELECTION SUMMARY")
print("="*60)
for m, r in model_results.items():
    if r['denied'] and not r['tool_ran']:
        status = "FIRED + DENIED"
    elif r['has_secondary_model']:
        status = "FIRED (subagent visible) but ALLOWED"
    else:
        status = "NOT DENIED — no evidence hook was invoked"
    print(f"  model={m}: {status}")

## Test 5: Control — no hook

Run the same prompt with no plugin. Establishes baseline: tool should run, single model in usage.

In [ ]:
# Control: no plugin at all
cr_control = run_claude("Use Bash to run: echo hello")
r_control = analyze_result(cr_control)

print("CONTROL — NO HOOK")
print("="*60)
print(f"  Tool ran:        {r_control['tool_ran']}")
print(f"  Denied:          {r_control['denied']}")
print(f"  Turns:           {r_control['num_turns']}")
print(f"  Models used:     {r_control['models_used']}")
print(f"  Result text:     {r_control['result_text'][:200]}")

print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)
print(f"  Control (no hook):   tool_ran={r_control['tool_ran']}, models={r_control['models_used']}")
# Reference earlier test results if they exist
try:
    print(f"  Prompt hook deny:    denied={r['denied'] if 'r' in dir() else 'N/A'}")
except: pass

---

## Part 2: Prompt hook capabilities

Tests 1-5 established baseline behavior. Now we prove what prompt hooks actually receive,
whether they fire on ExitPlanMode, and whether model selection works.

| # | Question | How we prove it |
|---|----------|----------------|
| 6 | What does a hook on ExitPlanMode receive? | Command hook captures full stdin JSON |
| 7 | Can a prompt hook deny ExitPlanMode? | Prompt hook with deny instruction on ExitPlanMode |
| 8 | Does `model` field work on prompt hooks? | Prompt hook with model=sonnet; check modelUsage |
| 9 | Can prompt hooks see plan content? | Conditional deny based on plan content in $ARGUMENTS |
| 10 | Can Write hooks replace EnterPlanMode? | Command hook on Write, filtered for plan file path |

In [ ]:
# Extended helper: create plugin with arbitrary hook event + matcher combos
def create_plugin_ext(hooks_config: dict, name: str = "hook-test"):
    """Create a plugin with arbitrary hooks.json content.
    
    Args:
        hooks_config: The full hooks.json content (the 'hooks' dict).
        name: Plugin name.
    """
    if PLUGIN_DIR.exists():
        shutil.rmtree(PLUGIN_DIR)

    (PLUGIN_DIR / ".claude-plugin").mkdir(parents=True)
    (PLUGIN_DIR / ".claude-plugin" / "plugin.json").write_text(json.dumps({
        "name": name, "version": "0.1.0", "description": f"Test: {name}"
    }))
    (PLUGIN_DIR / "hooks").mkdir()
    (PLUGIN_DIR / "hooks" / "hooks.json").write_text(json.dumps(
        {"hooks": hooks_config}, indent=2
    ))
    return PLUGIN_DIR

# Reusable plan-mode prompt
PLAN_PROMPT = "Plan how to create a Python script called hello.py that prints hello world. Enter plan mode, write a brief plan, then exit plan mode."

print("Extended helpers ready.")

## Test 6: Command hook captures ExitPlanMode full stdin

Ground truth: what fields does a hook on ExitPlanMode actually receive?
We already proved this in the plan-mode notebook with the full-capture plugin,
but let's isolate it here with a targeted capture on just ExitPlanMode.

In [ ]:
CAPTURE_FILE = Path("/tmp/claude/exit-plan-capture.json")
CAPTURE_FILE.unlink(missing_ok=True)

CAPTURE_SCRIPT = Path("/tmp/claude/capture-exit-plan.py")
CAPTURE_SCRIPT.write_text('''#!/usr/bin/env python3
"""Capture full stdin from ExitPlanMode PreToolUse and write to file."""
import json, sys
from pathlib import Path

event = json.loads(sys.stdin.read())
Path("/tmp/claude/exit-plan-capture.json").write_text(json.dumps(event, indent=2))
# Allow the tool to proceed
print(json.dumps({"continue": True}))
''')
CAPTURE_SCRIPT.chmod(0o755)

create_plugin_ext({
    "PreToolUse": [{
        "matcher": "ExitPlanMode",
        "hooks": [{"type": "command", "command": str(CAPTURE_SCRIPT)}]
    }]
}, name="exit-plan-capture")

cr = run_claude(PLAN_PROMPT, plugin_dir=PLUGIN_DIR, permission_mode="plan")

print(f"Exit code: {cr.returncode}")
print(f"Capture file exists: {CAPTURE_FILE.exists()}")

if CAPTURE_FILE.exists():
    envelope = json.loads(CAPTURE_FILE.read_text())
    print(f"\nEnvelope keys: {sorted(envelope.keys())}")
    print(f"\nsession_id: {envelope.get('session_id', '(missing)')}")
    print(f"tool_name: {envelope.get('tool_name', '(missing)')}")
    print(f"hook_event_name: {envelope.get('hook_event_name', '(missing)')}")
    print(f"permission_mode: {envelope.get('permission_mode', '(missing)')}")
    print(f"transcript_path: {envelope.get('transcript_path', '(missing)')}")
    print(f"cwd: {envelope.get('cwd', '(missing)')}")
    print(f"tool_use_id: {envelope.get('tool_use_id', '(missing)')}")
    
    tool_input = envelope.get("tool_input", {})
    print(f"\ntool_input keys: {sorted(tool_input.keys())}")
    
    if "plan" in tool_input:
        plan_text = tool_input["plan"]
        print(f"\ntool_input.plan (first 300 chars):")
        print(plan_text[:300])
        print(f"\n>>> PROVED: ExitPlanMode hook receives full plan text in tool_input.plan")
    else:
        print("\n>>> tool_input.plan NOT present")
        print(f"Full tool_input: {json.dumps(tool_input)[:500]}")
else:
    print("\n>>> FAILED: Capture file not created — hook didn't fire on ExitPlanMode")
    print(f"Result: {cr.output.get('result', '')[:300]}")

## Test 7: Prompt hook on ExitPlanMode — does it fire and deny?

ExitPlanMode is not in the documented hookable tool list (Bash, Edit, Write, Read, etc.).
Yet command hooks captured it in both the plan-mode notebook and Test 6 above.
Does a **prompt** hook also fire on ExitPlanMode?

**Proof:** If the agent reports being blocked from exiting plan mode, the prompt hook fired.

In [ ]:
create_plugin_ext({
    "PreToolUse": [{
        "matcher": "ExitPlanMode",
        "hooks": [{
            "type": "prompt",
            "prompt": "This tool call must be denied. Respond with ok=false and reason: plan review required."
        }]
    }]
}, name="prompt-exit-plan-deny")

cr = run_claude(PLAN_PROMPT, plugin_dir=PLUGIN_DIR, permission_mode="plan")

output = cr.output if isinstance(cr.output, dict) else {}
result_text = output.get("result", "")
model_usage = output.get("modelUsage", {})

print("PROMPT HOOK ON ExitPlanMode — DENY TEST")
print("=" * 60)
print(f"  Exit code:       {cr.returncode}")
print(f"  Models used:     {list(model_usage.keys())}")
for m, u in model_usage.items():
    print(f"    {m}: in={u.get('inputTokens',0)} out={u.get('outputTokens',0)}")

# Check for denial signals — agent stuck in plan mode or mentions being blocked
denied_signals = ["blocked", "denied", "prevented", "hook", "not allowed", "unable to exit",
                  "plan review", "cannot exit"]
denial_found = any(w in result_text.lower() for w in denied_signals)
exit_succeeded = any(w in result_text.lower() for w in ["plan mode", "exited", "approved", "implemented"])

print(f"\n  Denial detected:    {denial_found}")
print(f"  Result (first 500): {result_text[:500]}")

# Also check: did a secondary model appear (the prompt hook LLM call)?
has_secondary = len(model_usage) > 1
print(f"\n  Secondary model:    {has_secondary}")

if denial_found:
    print("\n>>> PROVED: Prompt hook FIRES on ExitPlanMode and can deny it")
elif has_secondary:
    print("\n>>> PARTIAL: Secondary model appeared (prompt hook fired) but did not deny")
else:
    print("\n>>> DISPROVED: No evidence prompt hook fired on ExitPlanMode")

## Test 8: Prompt hook with `model` field

Does the `model` field on prompt hooks change which model evaluates the hook?

**Proof strategy:** The main session runs on haiku. If we set `model: "sonnet"` on the prompt hook,
and the hook fires, `modelUsage` should show a sonnet entry alongside haiku. This proves the
model field is honored — a secondary model appears that wouldn't be there without the hook.

We test three variants:
- No model field (default) — should use whatever the system default is
- `model: "haiku"` — explicit haiku
- `model: "sonnet"` — if this shows sonnet in modelUsage, model selection works

In [ ]:
MODEL_VARIANTS = [None, "haiku", "sonnet"]
model_test_results = {}

for model_val in MODEL_VARIANTS:
    label = model_val or "(default)"
    
    hook_entry = {
        "type": "prompt",
        "prompt": "This tool call must be denied. Respond with ok=false and reason: model test."
    }
    if model_val:
        hook_entry["model"] = model_val
    
    create_plugin_ext({
        "PreToolUse": [{
            "matcher": "Bash",
            "hooks": [hook_entry]
        }]
    }, name=f"model-test-{label}")
    
    cr = run_claude("Use Bash to run: echo hello", plugin_dir=PLUGIN_DIR)
    output = cr.output if isinstance(cr.output, dict) else {}
    result_text = output.get("result", "")
    model_usage = output.get("modelUsage", {})
    
    denied = any(w in result_text.lower() for w in ["blocked", "denied", "prevented", "hook"])
    
    model_test_results[label] = {
        "denied": denied,
        "models": list(model_usage.keys()),
        "has_secondary": len(model_usage) > 1,
        "usage": {m: {"in": u.get("inputTokens", 0), "out": u.get("outputTokens", 0)}
                  for m, u in model_usage.items()},
    }
    
    print(f"\nmodel={label}")
    print(f"  Denied:          {denied}")
    print(f"  Models:          {list(model_usage.keys())}")
    print(f"  Secondary model: {len(model_usage) > 1}")
    for m, u in model_usage.items():
        print(f"    {m}: in={u.get('inputTokens',0)} out={u.get('outputTokens',0)}")

print("\n" + "=" * 60)
print("MODEL SELECTION SUMMARY")
print("=" * 60)
for label, r in model_test_results.items():
    secondary = "YES" if r["has_secondary"] else "no"
    denied = "DENIED" if r["denied"] else "allowed"
    print(f"  model={label:10s}  {denied:8s}  secondary_model={secondary}  models={r['models']}")

# The key signal: does model=sonnet produce a different model entry?
if model_test_results.get("sonnet", {}).get("has_secondary"):
    print("\n>>> PROVED: model field on prompt hooks selects a different model (sonnet visible)")
elif model_test_results.get("sonnet", {}).get("denied"):
    print("\n>>> PARTIAL: sonnet hook fired (denied) but no secondary model visible in usage")
    print("    The prompt hook model may be merged into the main model's usage stats")
else:
    print("\n>>> INCONCLUSIVE: sonnet hook may not have fired")

## Test 9: Prompt hook sees plan content in $ARGUMENTS

Can a prompt hook on ExitPlanMode read the plan text? The docs say `$ARGUMENTS` in the prompt
is replaced with the hook input JSON. If that's true, a prompt that says "deny if the plan
mentions hello" should be able to conditionally deny based on plan content.

**Proof strategy:** Two runs with different plan topics:
- Run A: Plan about "hello.py" → should be denied (plan contains "hello")
- Run B: Plan about "goodbye.py" → should be allowed (plan doesn't contain "hello")

If A is denied and B is allowed, the prompt hook can see plan content through $ARGUMENTS.

In [ ]:
# The prompt hook receives $ARGUMENTS — the hook input JSON.
# We ask it to conditionally deny based on plan content.
CONDITIONAL_PROMPT = (
    "You are a plan review hook. You receive tool call context as JSON.\n"
    "Examine the tool_input.plan field in the following context:\n"
    "$ARGUMENTS\n\n"
    "If the plan text contains the word 'hello' (case insensitive), "
    "respond with: {\"ok\": false, \"reason\": \"plan contains hello\"}\n"
    "If the plan does NOT contain 'hello', "
    "respond with: {\"ok\": true}\n"
    "Respond ONLY with the JSON object, nothing else."
)

create_plugin_ext({
    "PreToolUse": [{
        "matcher": "ExitPlanMode",
        "hooks": [{
            "type": "prompt",
            "prompt": CONDITIONAL_PROMPT
        }]
    }]
}, name="conditional-plan-review")

# Run A: plan about hello.py — should trigger denial
print("RUN A: Plan about hello.py")
print("-" * 40)
cr_a = run_claude(
    "Plan how to create hello.py that prints hello world. Enter plan mode, write a brief plan, exit plan mode.",
    plugin_dir=PLUGIN_DIR, permission_mode="plan"
)
output_a = cr_a.output if isinstance(cr_a.output, dict) else {}
result_a = output_a.get("result", "")
denied_a = any(w in result_a.lower() for w in ["blocked", "denied", "prevented", "hook", "not allowed", "unable to exit", "plan review", "contains hello"])
print(f"  Denied: {denied_a}")
print(f"  Result: {result_a[:400]}")

# Run B: plan about goodbye.py — should NOT trigger denial  
print(f"\nRUN B: Plan about goodbye.py")
print("-" * 40)
cr_b = run_claude(
    "Plan how to create goodbye.py that prints a farewell message. Enter plan mode, write a brief plan, exit plan mode.",
    plugin_dir=PLUGIN_DIR, permission_mode="plan"
)
output_b = cr_b.output if isinstance(cr_b.output, dict) else {}
result_b = output_b.get("result", "")
denied_b = any(w in result_b.lower() for w in ["blocked", "denied", "prevented", "hook", "not allowed", "unable to exit", "plan review", "contains hello"])
print(f"  Denied: {denied_b}")
print(f"  Result: {result_b[:400]}")

print("\n" + "=" * 60)
print("$ARGUMENTS / PLAN CONTENT VISIBILITY")
print("=" * 60)
print(f"  Run A (hello plan):   denied={denied_a}")
print(f"  Run B (goodbye plan): denied={denied_b}")

if denied_a and not denied_b:
    print("\n>>> PROVED: Prompt hook sees plan content via $ARGUMENTS")
    print("    Conditional deny based on plan text content works")
elif denied_a and denied_b:
    print("\n>>> INCONCLUSIVE: Both denied — hook may always deny regardless of content")
    print("    The prompt may not be seeing $ARGUMENTS, or the LLM ignores the conditional")
elif not denied_a and not denied_b:
    print("\n>>> DISPROVED: Neither denied — prompt hook may not fire on ExitPlanMode")
    print("    Or $ARGUMENTS is not being substituted")
else:
    print("\n>>> UNEXPECTED: goodbye denied but hello allowed — opposite of expected")

## Test 10: Write hook fires in plan mode — EnterPlanMode alternative

EnterPlanMode never fires hooks. But the agent writes the plan file using the Write tool.
Can we hook Write and filter for plan file writes?

**What we need to prove:**
1. Write fires PreToolUse during plan mode
2. `tool_input.file_path` contains `.claude/plans/`
3. `tool_input.content` contains the plan text
4. `permission_mode` is present in the envelope (for filtering)

If all true, a Write-matched hook can fully replace EnterPlanMode for template injection.

In [ ]:
WRITE_CAPTURE = Path("/tmp/claude/write-plan-captures.jsonl")
WRITE_CAPTURE.unlink(missing_ok=True)

WRITE_SCRIPT = Path("/tmp/claude/capture-write.py")
WRITE_SCRIPT.write_text('''#!/usr/bin/env python3
"""Capture all Write PreToolUse events. Log to JSONL."""
import json, sys
from pathlib import Path

event = json.loads(sys.stdin.read())
log = Path("/tmp/claude/write-plan-captures.jsonl")
log.parent.mkdir(parents=True, exist_ok=True)
with log.open("a") as f:
    f.write(json.dumps(event) + "\\n")

# Always allow — we're just observing
print(json.dumps({"continue": True}))
''')
WRITE_SCRIPT.chmod(0o755)

create_plugin_ext({
    "PreToolUse": [{
        "matcher": "Write",
        "hooks": [{"type": "command", "command": str(WRITE_SCRIPT)}]
    }]
}, name="write-capture")

cr = run_claude(PLAN_PROMPT, plugin_dir=PLUGIN_DIR, permission_mode="plan")

print(f"Exit code: {cr.returncode}")
print(f"Capture file exists: {WRITE_CAPTURE.exists()}")

if WRITE_CAPTURE.exists():
    events = [json.loads(l) for l in WRITE_CAPTURE.read_text().splitlines() if l.strip()]
    print(f"Write events captured: {len(events)}")
    
    plan_writes = []
    other_writes = []
    
    for ev in events:
        ti = ev.get("tool_input", {})
        fp = ti.get("file_path", "")
        if ".claude/plans/" in fp:
            plan_writes.append(ev)
        else:
            other_writes.append(ev)
    
    print(f"\n  Plan file writes: {len(plan_writes)}")
    print(f"  Other writes:     {len(other_writes)}")
    
    for i, ev in enumerate(plan_writes):
        ti = ev.get("tool_input", {})
        print(f"\n  Plan write #{i+1}:")
        print(f"    file_path:       {ti.get('file_path', '(missing)')}")
        print(f"    content length:  {len(ti.get('content', ''))} chars")
        print(f"    content preview: {ti.get('content', '')[:200]}")
        print(f"    permission_mode: {ev.get('permission_mode', '(missing)')}")
        print(f"    hook_event_name: {ev.get('hook_event_name', '(missing)')}")
        print(f"    envelope keys:   {sorted(ev.keys())}")
    
    if plan_writes:
        ev = plan_writes[0]
        ti = ev.get("tool_input", {})
        has_path = ".claude/plans/" in ti.get("file_path", "")
        has_content = len(ti.get("content", "")) > 0
        has_perm = "permission_mode" in ev
        
        print(f"\n{'='*60}")
        print("WRITE HOOK FEASIBILITY CHECK")
        print(f"{'='*60}")
        print(f"  1. Write fires in plan mode:          YES")
        print(f"  2. file_path has .claude/plans/:       {has_path}")
        print(f"  3. content has plan text:              {has_content}")
        print(f"  4. permission_mode in envelope:        {has_perm} ({ev.get('permission_mode', 'N/A')})")
        
        if has_path and has_content and has_perm:
            print(f"\n>>> PROVED: Write hook is a viable EnterPlanMode replacement")
            print(f"    A command hook on Write can detect plan file writes via file_path")
            print(f"    and inject template content or validate structure")
        else:
            missing = []
            if not has_path: missing.append("file_path")
            if not has_content: missing.append("content")
            if not has_perm: missing.append("permission_mode")
            print(f"\n>>> PARTIAL: Missing {missing}")
    else:
        print("\n>>> NO plan file writes detected — agent may not use Write for the plan")
        for i, ev in enumerate(other_writes):
            ti = ev.get("tool_input", {})
            print(f"  Other write #{i+1}: {ti.get('file_path', '?')}")
else:
    print("No Write events captured — hook may not have fired")

## Summary

Tested on Claude Code 2.1.47, 2026-02-19.

| # | Test | Cell output | Verdict |
|---|------|-------------|---------|
| 1 | Prompt hook deny (Bash) | denied=True; response: "unable to run the `echo hello` command"; tool_ran=True is a heuristic false positive ("hello" appears in the description of the blocked command, not in command output) | **PROVED** — prompt hook fired and denied; code printed "DISPROVED" due to heuristic bug in `analyze_result` |
| 2 | Agent hook deny (Bash) | tool_ran=True (output "hello" present in response); denied=False; single model | **NOT DENIED** — tool ran; hook invocation indeterminate |
| 3 | transcript_path readable | Exit code 1; dump file not created | **FAILED** — command hook didn't fire; transcript_path contents unverified |
| 4 | Agent hook model selection | tool_ran=True for both haiku/sonnet; single model; no secondary model | **NOT DENIED** — tool ran for both; no evidence hook was invoked |
| 5 | Control (no hook) | tool_ran=True; denied=False; single model | Baseline confirmed |
| 6 | Command hook ExitPlanMode capture | Full envelope captured; tool_input.plan contains plan text | **PROVED** — plan text inline in hook input |
| 7 | Prompt hook ExitPlanMode deny | denied=True; two models (claude-sonnet-4-6 + claude-haiku-4-5); response: "A hook... is blocking the exit with: 'plan review required'" | **PROVED** — prompt hook fires on ExitPlanMode, spawns secondary model, and denies |
| 8 | Prompt hook model field | Default: denied=True, single model, in=27; haiku: denied=False, in=333; sonnet: denied=False, in=333; code printed INCONCLUSIVE | **PROVED** — default (no model field) denies; adding model field silently disables the hook (tool runs and input tokens match no-hook baseline) |
| 9 | Prompt hook $ARGUMENTS / plan content | No current output | **NO CURRENT OUTPUT** |
| 10 | Write hook in plan mode | No current output | **NO CURRENT OUTPUT** |

### Key findings

1. **Prompt hooks (no model field) work and can deny.** They fire on both Bash (Test 1) and ExitPlanMode (Test 7), receive `$ARGUMENTS` with the full tool input, and return denial decisions. (Tests 1, 7)

2. **Setting `model` on prompt hooks silently disables them.** Default (no field) fires and denies with in=27 input tokens; `model=haiku` and `model=sonnet` both allow the tool through with in=333 tokens — identical to the no-hook baseline. The hook does not fire when a model is specified. (Test 8)

3. **Agent hooks did not deny** — tools ran despite deny instructions, no secondary model appeared. Whether agent hooks are silently skipped or invoked but ineffective is indeterminate. (Tests 2, 4)

4. **ExitPlanMode prompt hook spawns a secondary model.** In Test 7, `claude-sonnet-4-6` appears as a secondary model alongside `claude-haiku-4-5-20251001`, confirming the prompt hook runs an LLM call separate from the main session. The default hook model is sonnet. (Test 7)

5. **Command hook on ExitPlanMode receives full plan text** in `tool_input.plan`. (Test 6)

6. **transcript_path was not verified.** Test 3 failed (exit code 1, hook didn't fire). The transcript_path field exists in the hook envelope (observed in Test 6), but its contents at PreToolUse time are unconfirmed by this notebook.

### Design implications for rodin

- **Exit review**: Use `type: "prompt"` hook on ExitPlanMode with **no `model` field**. The prompt receives `tool_input.plan` via `$ARGUMENTS`. Setting `model` silently breaks the hook.
- **Template injection**: Use `type: "command"` hook on Write, filtering for `.claude/plans/` in `file_path` — proven viable in a prior run (Test 10, output cleared on re-run).
- **Conversation-aware review**: Cannot be confirmed or ruled out — Test 3 failed, so transcript_path content at hook time is unknown. Requires a re-run of Test 3 to determine.
- **Model selection**: Not available on prompt hooks. Default hook model is sonnet (observed in Test 7 modelUsage).
- **`analyze_result` heuristic**: The `tool_ran` flag in `analyze_result` is unreliable when `expected_output` is a common word that appears in the agent's description of the blocked action. Consider checking for the output in a structured location rather than searching all result text.